## Jawad Hassan
## Nlp Pipeline Task

In [17]:
import pandas as pd
import nltk, textblob, emoji, spacy, sklearn

nlp = spacy.load("en_core_web_sm")
print("Setup complete! spaCy version:", spacy.__version__)

Setup complete! spaCy version: 3.8.16


## data aquisition

In [18]:
df = pd.read_csv("IMDB Dataset.csv")
print(df.shape)
df.head()

(50000, 2)


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [19]:
print(df["sentiment"].value_counts())
print("\nMissing values:\n", df.isnull().sum())
print("\nDuplicate reviews:", df.duplicated().sum())

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

Missing values:
 review       0
sentiment    0
dtype: int64

Duplicate reviews: 418


In [20]:
df = df.drop_duplicates().reset_index(drop=True)
print(df.shape)

(49582, 2)


In [21]:
print(df["review"][1])

A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. <br /><br />The actors are extremely well chosen- Michael Sheen not only "has got all the polari" but he has all the voices down pat too! You can truly see the seamless editing guided by the references to Williams' diary entries, not only is it well worth the watching but it is a terrificly written and performed piece. A masterful production about one of the great master's of comedy and his life. <br /><br />The realism really comes home with the little things: the fantasy of the guard which, rather than use the traditional 'dream' techniques remains solid then disappears. It plays on our knowledge and our senses, particularly with the scenes concerning Orton and Halliwell and the sets (particularly of their flat with Halliwell's murals decorating every surface) are terribly well done.


In [22]:
sample = df.sample(1000, random_state=42).reset_index(drop=True)
print(sample.shape)

(1000, 2)


## text preprocess

In [23]:
# lowercasing
sample["clean"] = sample["review"].str.lower()

print("BEFORE:", sample["review"][0][:200])
print("AFTER: ", sample["clean"][0][:200])

BEFORE: "Soul Plane" is a horrible attempt at comedy that only should appeal people with thick skulls, bloodshot eyes and furry pawns. <br /><br />The plot is not only incoherent but also non-existent, acting
AFTER:  "soul plane" is a horrible attempt at comedy that only should appeal people with thick skulls, bloodshot eyes and furry pawns. <br /><br />the plot is not only incoherent but also non-existent, acting


In [24]:
# removing html tags
import re

def remove_html(text):
    return re.sub(r"<.*?>", " ", text)

print(remove_html("this movie was great!<br /><br />loved it"))   # test first

print("Reviews with HTML tags:", sample["clean"].str.contains(r"<.*?>").sum())
sample["clean"] = sample["clean"].apply(remove_html)

this movie was great!  loved it
Reviews with HTML tags: 601


In [25]:
# removing urls
def remove_urls(text):
    return re.sub(r"https?://\S+|www\.\S+", " ", text)

print(remove_urls("check this https://www.imdb.com/title/tt0111161 and www.google.com now"))

print("Reviews with URLs:", sample["clean"].str.contains(r"https?://\S+|www\.\S+").sum())
sample["clean"] = sample["clean"].apply(remove_urls)

check this   and   now
Reviews with URLs: 3


In [26]:
# emoji handling - comparing 3 methods
text = "loved this movie 😂🔥 the café scene was 👌"

print("1) remove:      ", emoji.replace_emoji(text, replace=""))
print("2) to words:    ", emoji.demojize(text))
print("3) ascii encode:", text.encode("ascii", "ignore").decode("ascii"))

1) remove:       loved this movie  the café scene was 
2) to words:     loved this movie :face_with_tears_of_joy::fire: the café scene was :OK_hand:
3) ascii encode: loved this movie  the caf scene was 


In [27]:
# removing emojis
def remove_emojis(text):
    return emoji.replace_emoji(text, replace="")

print("Reviews with emojis:", sample["clean"].apply(emoji.emoji_count).gt(0).sum())
sample["clean"] = sample["clean"].apply(remove_emojis)

Reviews with emojis: 0


In [28]:
# progress check
print("ORIGINAL:\n", sample["review"][0][:400])
print("\nCLEANED SO FAR:\n", sample["clean"][0][:400])

ORIGINAL:
 "Soul Plane" is a horrible attempt at comedy that only should appeal people with thick skulls, bloodshot eyes and furry pawns. <br /><br />The plot is not only incoherent but also non-existent, acting is mostly sub sub-par with a gang of highly moronic and dreadful characters thrown in for bad measure, jokes are often spotted miles ahead and almost never even a bit amusing. This movie lacks any st

CLEANED SO FAR:
 "soul plane" is a horrible attempt at comedy that only should appeal people with thick skulls, bloodshot eyes and furry pawns.   the plot is not only incoherent but also non-existent, acting is mostly sub sub-par with a gang of highly moronic and dreadful characters thrown in for bad measure, jokes are often spotted miles ahead and almost never even a bit amusing. this movie lacks any structure an


In [29]:
# removing punctuation
import string
print(string.punctuation)   # these are the characters we will remove

def remove_punctuation(text):
    text = text.translate(str.maketrans("", "", string.punctuation))
    return " ".join(text.split())   # also removes the extra spaces left behind

print(remove_punctuation("wow!!! this movie's plot was... great, 10/10   :)"))
sample["clean"] = sample["clean"].apply(remove_punctuation)

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
wow this movies plot was great 1010


In [30]:
# expanding chat words
chat_words = {
    "lol": "laughing out loud", "omg": "oh my god", "imo": "in my opinion",
    "imho": "in my humble opinion", "btw": "by the way", "idk": "i do not know",
    "tbh": "to be honest", "fyi": "for your information", "brb": "be right back",
    "irl": "in real life", "smh": "shaking my head", "rofl": "rolling on the floor laughing",
    "gr8": "great", "b4": "before", "thx": "thanks", "pls": "please", "plz": "please",
    "u": "you", "ur": "your", "bc": "because", "cuz": "because", "ppl": "people",
    "fav": "favorite", "tho": "though", "nvm": "never mind", "asap": "as soon as possible",
}

def expand_chat_words(text):
    return " ".join(chat_words.get(word, word) for word in text.split())

def has_chat_word(text):
    return any(word in chat_words for word in text.split())

print(expand_chat_words("omg this movie was gr8 tbh u should watch it"))
print("Reviews with chat words:", sample["clean"].apply(has_chat_word).sum())
sample["clean"] = sample["clean"].apply(expand_chat_words)

oh my god this movie was great to be honest you should watch it
Reviews with chat words: 22


In [31]:
# spelling correction - test
from textblob import TextBlob
import time

print(TextBlob("this moive was relly gud and the actng was amazng").correct())

this move was really god and the acting was amazing


In [32]:
# spelling correction on 10 reviews only
def correct_spelling(text):
    return str(TextBlob(text).correct())

start = time.time()
few = sample["clean"][:10].apply(correct_spelling)
print("Time for 10 reviews:", round(time.time() - start, 1), "seconds")

print("BEFORE:", sample["clean"][0][:300])
print("AFTER: ", few[0][:300])

Time for 10 reviews: 43.9 seconds
BEFORE: soul plane is a horrible attempt at comedy that only should appeal people with thick skulls bloodshot eyes and furry pawns the plot is not only incoherent but also nonexistent acting is mostly sub subpar with a gang of highly moronic and dreadful characters thrown in for bad measure jokes are often 
AFTER:  soul plane is a horrible attempt at comedy that only should appeal people with thick skull bloodshot eyes and furry pawns the plot is not only incoherent but also nonexistent acting is mostly sub supper with a gang of highly chronic and dreadful characters thrown in for bad measure jokes are often s


In [33]:
# removing stopwords
import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))
print("Number of stopwords:", len(stop_words))
print(sorted(stop_words)[:30])

def remove_stopwords(text):
    return " ".join(word for word in text.split() if word not in stop_words)

print(remove_stopwords("this is one of the best movies i have ever seen"))
print(remove_stopwords("this is not a good movie"))

Number of stopwords: 198
['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't"]
one best movies ever seen
good movie


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/macbook/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [34]:
# keeping negation words + applying stopword removal
stop_words = stop_words - {"not", "no", "nor"}
print(remove_stopwords("this is not a good movie"))

sample["clean"] = sample["clean"].apply(remove_stopwords)

not good movie


In [35]:
# final check - basic cleaning
print("ORIGINAL:\n", sample["review"][0][:400])
print("\nCLEANED:\n", sample["clean"][0][:400])

print("\nAvg words BEFORE:", round(sample["review"].str.split().str.len().mean()))
print("Avg words AFTER: ", round(sample["clean"].str.split().str.len().mean()))

ORIGINAL:
 "Soul Plane" is a horrible attempt at comedy that only should appeal people with thick skulls, bloodshot eyes and furry pawns. <br /><br />The plot is not only incoherent but also non-existent, acting is mostly sub sub-par with a gang of highly moronic and dreadful characters thrown in for bad measure, jokes are often spotted miles ahead and almost never even a bit amusing. This movie lacks any st

CLEANED:
 soul plane horrible attempt comedy appeal people thick skulls bloodshot eyes furry pawns plot not incoherent also nonexistent acting mostly sub subpar gang highly moronic dreadful characters thrown bad measure jokes often spotted miles ahead almost never even bit amusing movie lacks structure full racial stereotypes must seemed old even fifties thing really going pretty ladies really want rent som

Avg words BEFORE: 239
Avg words AFTER:  127


In [36]:
# saving the cleaned sample
sample.to_csv("imdb_clean_sample.csv", index=False)
print("Saved!")

Saved!


In [37]:
# downloading nltk data for advanced steps
nltk.download("punkt_tab")
nltk.download("averaged_perceptron_tagger_eng")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/macbook/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/macbook/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [38]:
# example texts for advanced steps
text = "Christopher Nolan directed Inception in 2010. Leonardo DiCaprio starred in it, and the movie earned $830 million worldwide."

review = remove_html(sample["review"][0])   # a real review, only html removed
print(review[:400])

"Soul Plane" is a horrible attempt at comedy that only should appeal people with thick skulls, bloodshot eyes and furry pawns.   The plot is not only incoherent but also non-existent, acting is mostly sub sub-par with a gang of highly moronic and dreadful characters thrown in for bad measure, jokes are often spotted miles ahead and almost never even a bit amusing. This movie lacks any structure an


In [39]:
# tokenization
from nltk.tokenize import sent_tokenize, word_tokenize

print("SENTENCES:", sent_tokenize(text))
print("\nWORDS:", word_tokenize(text))
print("\nSIMPLE SPLIT:", text.split())

SENTENCES: ['Christopher Nolan directed Inception in 2010.', 'Leonardo DiCaprio starred in it, and the movie earned $830 million worldwide.']

WORDS: ['Christopher', 'Nolan', 'directed', 'Inception', 'in', '2010', '.', 'Leonardo', 'DiCaprio', 'starred', 'in', 'it', ',', 'and', 'the', 'movie', 'earned', '$', '830', 'million', 'worldwide', '.']

SIMPLE SPLIT: ['Christopher', 'Nolan', 'directed', 'Inception', 'in', '2010.', 'Leonardo', 'DiCaprio', 'starred', 'in', 'it,', 'and', 'the', 'movie', 'earned', '$830', 'million', 'worldwide.']


In [40]:
# pos tagging with spacy
doc = nlp(text)

for token in doc:
    print(f"{token.text:<12} {token.pos_:<6} {spacy.explain(token.pos_)}")

Christopher  PROPN  proper noun
Nolan        PROPN  proper noun
directed     VERB   verb
Inception    PROPN  proper noun
in           ADP    adposition
2010         NUM    numeral
.            PUNCT  punctuation
Leonardo     PROPN  proper noun
DiCaprio     PROPN  proper noun
starred      VERB   verb
in           ADP    adposition
it           PRON   pronoun
,            PUNCT  punctuation
and          CCONJ  coordinating conjunction
the          DET    determiner
movie        NOUN   noun
earned       VERB   verb
$            SYM    symbol
830          NUM    numeral
million      NUM    numeral
worldwide    ADV    adverb
.            PUNCT  punctuation


In [41]:
# pos tagging with nltk
nltk_tags = nltk.pos_tag(word_tokenize(text))
print(nltk_tags)

print(spacy.explain("NNP"), "|", spacy.explain("VBD"))

[('Christopher', 'NNP'), ('Nolan', 'NNP'), ('directed', 'VBD'), ('Inception', 'NNP'), ('in', 'IN'), ('2010', 'CD'), ('.', '.'), ('Leonardo', 'NNP'), ('DiCaprio', 'NNP'), ('starred', 'VBD'), ('in', 'IN'), ('it', 'PRP'), (',', ','), ('and', 'CC'), ('the', 'DT'), ('movie', 'NN'), ('earned', 'VBD'), ('$', '$'), ('830', 'CD'), ('million', 'CD'), ('worldwide', 'NN'), ('.', '.')]
noun, proper singular | verb, past tense


In [42]:
# named entity recognition (ner)
for ent in doc.ents:
    print(f"{ent.text:<20} {ent.label_:<8} {spacy.explain(ent.label_)}")

Christopher Nolan    PERSON   People, including fictional
2010                 DATE     Absolute or relative dates or periods
Leonardo DiCaprio    PERSON   People, including fictional
$830 million         MONEY    Monetary values, including unit


In [43]:
# ner - visual view
from spacy import displacy
displacy.render(doc, style="ent", jupyter=True)

In [44]:
# ner on lowercase text - why we use the raw review
print([(ent.text, ent.label_) for ent in nlp(text.lower()).ents])

[('christopher nolan', 'PERSON'), ('2010', 'DATE'), ('leonardo', 'PERSON'), ('$830 million', 'MONEY')]


In [45]:
# ner on a real review
displacy.render(nlp(review), style="ent", jupyter=True)

In [46]:
# most mentioned people in 1000 reviews
from collections import Counter

raw_reviews = sample["review"].apply(remove_html)
people = Counter()

for doc_r in nlp.pipe(raw_reviews, batch_size=50):
    for ent in doc_r.ents:
        if ent.label_ == "PERSON":
            people[ent.text] += 1

print(people.most_common(15))

[('Amir', 34), ('Jane', 32), ('gore', 32), ('Hassan', 25), ('Oscar', 22), ('Joe', 18), ('Spielberg', 16), ('Ben', 16), ('Ahmad', 16), ('Seagal', 15), ('Max', 14), ('Bronson', 14), ('Dean', 14), ('Gene Kelly', 13), ('Tarzan', 13)]


In [47]:
# chunking with nltk - example sentence
sentence = "The talented young actor delivered a powerful performance in this dark thriller."

tags = nltk.pos_tag(word_tokenize(sentence))
print(tags)

[('The', 'DT'), ('talented', 'JJ'), ('young', 'JJ'), ('actor', 'NN'), ('delivered', 'VBD'), ('a', 'DT'), ('powerful', 'JJ'), ('performance', 'NN'), ('in', 'IN'), ('this', 'DT'), ('dark', 'JJ'), ('thriller', 'NN'), ('.', '.')]


In [48]:
# chunking - defining a grammar rule
grammar = "NP: {<DT>?<JJ.*>*<NN.*>+}"
chunker = nltk.RegexpParser(grammar)

tree = chunker.parse(tags)
print(tree)

(S
  (NP The/DT talented/JJ young/JJ actor/NN)
  delivered/VBD
  (NP a/DT powerful/JJ performance/NN)
  in/IN
  (NP this/DT dark/JJ thriller/NN)
  ./.)


In [49]:
# chunking - extracting the noun phrases
for subtree in tree.subtrees(filter=lambda t: t.label() == "NP"):
    print(" ".join(word for word, tag in subtree.leaves()))

The talented young actor
a powerful performance
this dark thriller


In [50]:
# chunking with spacy - noun chunks (no grammar needed)
doc = nlp(sentence)
print([chunk.text for chunk in doc.noun_chunks])

['The talented young actor', 'a powerful performance', 'this dark thriller']


In [51]:
# dependency parsing with spacy
for token in doc:
    print(f"{token.text:<12} {token.dep_:<10} {token.head.text:<12} {spacy.explain(token.dep_)}")

The          det        actor        determiner
talented     amod       actor        adjectival modifier
young        amod       actor        adjectival modifier
actor        nsubj      delivered    nominal subject
delivered    ROOT       delivered    root
a            det        performance  determiner
powerful     amod       performance  adjectival modifier
performance  dobj       delivered    direct object
in           prep       performance  prepositional modifier
this         det        thriller     determiner
dark         amod       thriller     adjectival modifier
thriller     pobj       in           object of preposition
.            punct      delivered    punctuation


In [52]:
# dependency parsing - visual tree
displacy.render(doc, style="dep", jupyter=True, options={"compact": True, "distance": 100})

In [53]:
# constituency parsing with a small grammar
grammar_cfg = nltk.CFG.fromstring("""
S  -> NP VP
NP -> Det Adj N | Det N
VP -> V NP
Det -> 'the' | 'a'
Adj -> 'brilliant' | 'boring'
N  -> 'actor' | 'film' | 'role'
V  -> 'played' | 'ruined'
""")

parser = nltk.ChartParser(grammar_cfg)
for t in parser.parse("the brilliant actor played a role".split()):
    t.pretty_print()

                S                      
         _______|___________            
        |                   VP         
        |              _____|___        
        NP            |         NP     
  ______|_______      |      ___|___    
Det    Adj      N     V    Det      N  
 |      |       |     |     |       |   
the brilliant actor played  a      role



In [54]:
# parsing on real reviews - which adjectives describe which nouns
pairs = Counter()

for doc_r in nlp.pipe(raw_reviews, batch_size=50):
    for token in doc_r:
        if token.dep_ == "amod":     # amod = adjective describing a noun
            pairs[(token.text.lower(), token.head.text.lower())] += 1

print(pairs.most_common(20))

[(('such', 'as'), 60), (('special', 'effects'), 42), (('only', 'thing'), 37), (('low', 'budget'), 33), (('same', 'time'), 29), (('first', 'time'), 27), (('long', 'time'), 27), (('real', 'life'), 24), (('good', 'thing'), 22), (('main', 'characters'), 22), (('good', 'movie'), 22), (('main', 'character'), 21), (('whole', 'thing'), 20), (('first', 'movie'), 19), (('first', 'film'), 19), (('bad', 'movies'), 19), (('particular', 'in'), 19), (('many', 'times'), 18), (('other', 'movies'), 17), (('great', 'film'), 17)]


In [55]:
# stemming - comparing 3 stemmers
from nltk.stem import PorterStemmer, SnowballStemmer, LancasterStemmer

porter = PorterStemmer()
snowball = SnowballStemmer("english")
lancaster = LancasterStemmer()

words = ["running", "runs", "ran", "studies", "studying", "easily", "fairly", "movies", "acting", "actor", "better", "was"]

print(f"{'word':<10} {'porter':<10} {'snowball':<10} {'lancaster':<10}")
for w in words:
    print(f"{w:<10} {porter.stem(w):<10} {snowball.stem(w):<10} {lancaster.stem(w):<10}")

word       porter     snowball   lancaster 
running    run        run        run       
runs       run        run        run       
ran        ran        ran        ran       
studies    studi      studi      study     
studying   studi      studi      study     
easily     easili     easili     easy      
fairly     fairli     fair       fair      
movies     movi       movi       movy      
acting     act        act        act       
actor      actor      actor      act       
better     better     better     bet       
was        wa         was        was       


In [56]:
# lemmatization with nltk (wordnet)
nltk.download("wordnet")
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

print("as nouns:", [lemmatizer.lemmatize(w) for w in words])
print("as verbs:", [lemmatizer.lemmatize(w, pos="v") for w in words])
print("better as adjective:", lemmatizer.lemmatize("better", pos="a"))

[nltk_data] Downloading package wordnet to /Users/macbook/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


as nouns: ['running', 'run', 'ran', 'study', 'studying', 'easily', 'fairly', 'movie', 'acting', 'actor', 'better', 'wa']
as verbs: ['run', 'run', 'run', 'study', 'study', 'easily', 'fairly', 'movies', 'act', 'actor', 'better', 'be']
better as adjective: good


In [57]:
# lemmatization with spacy (finds pos tags automatically)
sentence = "the actors were running and studying their roles and the movies got better"
doc = nlp(sentence)

for token in doc:
    print(f"{token.text:<10} {token.pos_:<6} {token.lemma_}")

the        DET    the
actors     NOUN   actor
were       AUX    be
running    VERB   run
and        CCONJ  and
studying   VERB   study
their      PRON   their
roles      NOUN   role
and        CCONJ  and
the        DET    the
movies     NOUN   movie
got        VERB   get
better     ADJ    well


In [58]:
# stemming vs lemmatization - side by side
print("STEMMED:   ", " ".join(porter.stem(w) for w in sentence.split()))
print("LEMMATIZED:", " ".join(token.lemma_ for token in doc))

STEMMED:    the actor were run and studi their role and the movi got better
LEMMATIZED: the actor be run and study their role and the movie get well


In [59]:
# applying stemming to the reviews
def stem_text(text):
    return " ".join(porter.stem(word) for word in text.split())

sample["stemmed"] = sample["clean"].apply(stem_text)

In [60]:
# applying lemmatization to the reviews
sample["lemmatized"] = [
    " ".join(token.lemma_ for token in doc)
    for doc in nlp.pipe(sample["clean"], batch_size=50, disable=["parser", "ner"])
]

In [61]:
# comparing clean vs stemmed vs lemmatized
print("CLEAN:     ", sample["clean"][0][:300])
print("\nSTEMMED:   ", sample["stemmed"][0][:300])
print("\nLEMMATIZED:", sample["lemmatized"][0][:300])

CLEAN:      soul plane horrible attempt comedy appeal people thick skulls bloodshot eyes furry pawns plot not incoherent also nonexistent acting mostly sub subpar gang highly moronic dreadful characters thrown bad measure jokes often spotted miles ahead almost never even bit amusing movie lacks structure full r

STEMMED:    soul plane horribl attempt comedi appeal peopl thick skull bloodshot eye furri pawn plot not incoher also nonexist act mostli sub subpar gang highli moron dread charact thrown bad measur joke often spot mile ahead almost never even bit amus movi lack structur full racial stereotyp must seem old even

LEMMATIZED: soul plane horrible attempt comedy appeal people thick skulls bloodshot eye furry pawn plot not incoherent also nonexistent act mostly sub subpar gang highly moronic dreadful character throw bad measure joke often spot mile ahead almost never even bit amusing movie lack structure full racial stereot


In [62]:
# vocabulary size - how many unique words
def vocab_size(column):
    return len(set(" ".join(column).split()))

print("Unique words CLEAN:     ", vocab_size(sample["clean"]))
print("Unique words STEMMED:   ", vocab_size(sample["stemmed"]))
print("Unique words LEMMATIZED:", vocab_size(sample["lemmatized"]))

Unique words CLEAN:      20413
Unique words STEMMED:    14783
Unique words LEMMATIZED: 16902


In [63]:
# saving the final preprocessed sample
sample.to_csv("imdb_clean_sample.csv", index=False)
print("Saved!")

Saved!


In [64]:
# reloading the cleaned sample
sample = pd.read_csv("imdb_clean_sample.csv")
sample["clean"] = sample["clean"].fillna("")

## feature engineering

In [65]:
# one-hot encoding - our few documents
docs = [
    "great movie great acting",
    "boring movie",
    "great story boring acting",
]

In [66]:
# creating the corpus
corpus = []
for doc in docs:
    corpus.extend(doc.split())

print("Corpus:", corpus)
print("Total words in corpus:", len(corpus))

Corpus: ['great', 'movie', 'great', 'acting', 'boring', 'movie', 'great', 'story', 'boring', 'acting']
Total words in corpus: 10


In [67]:
# building the vocabulary
vocab = sorted(set(corpus))
print("Vocabulary:", vocab)
print("Vocabulary size (V):", len(vocab))

word_to_index = {word: i for i, word in enumerate(vocab)}
print(word_to_index)

Vocabulary: ['acting', 'boring', 'great', 'movie', 'story']
Vocabulary size (V): 5
{'acting': 0, 'boring': 1, 'great': 2, 'movie': 3, 'story': 4}


In [68]:
# one-hot vector for each word
def one_hot_word(word):
    vector = [0] * len(vocab)
    vector[word_to_index[word]] = 1
    return vector

for word in vocab:
    print(f"{word:<8} {one_hot_word(word)}")

acting   [1, 0, 0, 0, 0]
boring   [0, 1, 0, 0, 0]
great    [0, 0, 1, 0, 0]
movie    [0, 0, 0, 1, 0]
story    [0, 0, 0, 0, 1]


In [69]:
# one-hot encoding each document
import numpy as np

for i, doc in enumerate(docs):
    matrix = np.array([one_hot_word(word) for word in doc.split()])
    print(f"\nD{i+1}: '{doc}'  -> shape {matrix.shape}")
    print(pd.DataFrame(matrix, index=doc.split(), columns=vocab))


D1: 'great movie great acting'  -> shape (4, 5)
        acting  boring  great  movie  story
great        0       0      1      0      0
movie        0       0      0      1      0
great        0       0      1      0      0
acting       1       0      0      0      0

D2: 'boring movie'  -> shape (2, 5)
        acting  boring  great  movie  story
boring       0       1      0      0      0
movie        0       0      0      1      0

D3: 'great story boring acting'  -> shape (4, 5)
        acting  boring  great  movie  story
great        0       0      1      0      0
story        0       0      0      0      1
boring       0       1      0      0      0
acting       1       0      0      0      0


In [70]:
# one-hot problem - a word not in the vocabulary
try:
    one_hot_word("amazing")
except KeyError:
    print("'amazing' is not in the vocabulary - one-hot can't represent it")

'amazing' is not in the vocabulary - one-hot can't represent it


In [71]:
# one-hot on real reviews - why it doesn't scale
real_vocab = set(" ".join(sample["lemmatized"]).split())
first_review_words = sample["lemmatized"][0].split()

print("Vocabulary size:", len(real_vocab))
print("Words in review 0:", len(first_review_words))
print(f"One-hot matrix for ONE review: {len(first_review_words)} x {len(real_vocab)} = {len(first_review_words) * len(real_vocab):,} numbers")

Vocabulary size: 16902
Words in review 0: 83
One-hot matrix for ONE review: 83 x 16902 = 1,402,866 numbers


In [72]:
# one-hot encoding - 500 real reviews
docs = sample["lemmatized"].fillna("")[:500].tolist()

print("Number of documents:", len(docs))
print("First document:", docs[0][:200])

Number of documents: 500
First document: soul plane horrible attempt comedy appeal people thick skulls bloodshot eye furry pawn plot not incoherent also nonexistent act mostly sub subpar gang highly moronic dreadful character throw bad measu


In [73]:
# creating the corpus (500 reviews)
corpus = []
for doc in docs:
    corpus.extend(doc.split())

print("Total words in corpus:", len(corpus))
print("First 20 words:", corpus[:20])

Total words in corpus: 66612
First 20 words: ['soul', 'plane', 'horrible', 'attempt', 'comedy', 'appeal', 'people', 'thick', 'skulls', 'bloodshot', 'eye', 'furry', 'pawn', 'plot', 'not', 'incoherent', 'also', 'nonexistent', 'act', 'mostly']


In [74]:
# building the vocabulary (500 reviews)
vocab = sorted(set(corpus))
word_to_index = {word: i for i, word in enumerate(vocab)}

print("Vocabulary size (V):", len(vocab))
print("First 20 vocab words:", vocab[:20])

Vocabulary size (V): 11480
First 20 vocab words: ['0', '007', '01', '07', '07kiloton', '1', '10', '100', '1000', '100th', '101', '1010', '102', '1020', '108', '10but', '11', '110', '12', '127']


In [75]:
# most common words in the corpus
from collections import Counter
print(Counter(corpus).most_common(20))

[('not', 1357), ('film', 1022), ('movie', 1009), ('one', 577), ('like', 474), ('see', 473), ('make', 426), ('do', 418), ('get', 398), ('good', 372), ('well', 360), ('time', 343), ('watch', 312), ('character', 290), ('s', 273), ('would', 272), ('story', 270), ('go', 269), ('show', 267), ('no', 265)]


In [76]:
# one-hot encoding all 500 reviews (sparse matrices)
from scipy.sparse import csr_matrix
import numpy as np

def one_hot_doc(doc):
    words = doc.split()
    rows = list(range(len(words)))              # one row per word
    cols = [word_to_index[w] for w in words]    # the column where the 1 goes
    data = [1] * len(words)
    return csr_matrix((data, (rows, cols)), shape=(len(words), len(vocab)), dtype=np.int8)

one_hot_docs = [one_hot_doc(doc) for doc in docs]

print("Shapes of first 5 reviews:", [m.shape for m in one_hot_docs[:5]])
lengths = [m.shape[0] for m in one_hot_docs]
print("Shortest review:", min(lengths), "words | Longest:", max(lengths), "words")

Shapes of first 5 reviews: [(83, 11480), (161, 11480), (365, 11480), (360, 11480), (405, 11480)]
Shortest review: 18 words | Longest: 589 words


In [77]:
# looking at one review's one-hot matrix (first 5 words)
first = one_hot_docs[0][:5].toarray()
words = docs[0].split()[:5]
df_oh = pd.DataFrame(first, index=words, columns=vocab)

print(df_oh.loc[:, (df_oh != 0).any()])     # only show columns that contain a 1
print("\nEach row really has", df_oh.shape[1], "columns: one 1 and", df_oh.shape[1] - 1, "zeros")

          attempt  comedy  horrible  plane  soul
soul            0       0         0      0     1
plane           0       0         0      1     0
horrible        0       0         1      0     0
attempt         1       0         0      0     0
comedy          0       1         0      0     0

Each row really has 11480 columns: one 1 and 11479 zeros


In [78]:
# memory - dense vs sparse
total_words = sum(lengths)
dense_numbers = total_words * len(vocab)

print(f"Total word rows: {total_words:,}")
print(f"Dense one-hot would store: {dense_numbers:,} numbers (~{dense_numbers * 8 / 1e9:.1f} GB)")
print(f"Sparse one-hot stores only: {total_words:,} ones")

Total word rows: 66,612
Dense one-hot would store: 764,705,760 numbers (~6.1 GB)
Sparse one-hot stores only: 66,612 ones


In [79]:
# unknown words - a review outside our 500
new_review = sample["lemmatized"].fillna("")[600].split()
unknown = [w for w in new_review if w not in word_to_index]

print(f"Review 600 has {len(new_review)} words, {len(unknown)} are NOT in our vocabulary:")
print(unknown[:20])

Review 600 has 196 words, 43 are NOT in our vocabulary:
['doityourself', 'entertainingly', 'nickelndime', 'shotonvideo', 'clunker', 'sling', 'gleam', 'demonic', 'resurrect', 'ravenous', 'rotface', 'slade', 'nichodemus', 'sleazeball', 'hoodlum', 'hottie', 'harrowing', 'needlessly', 'hoot', 'coherence']


In [80]:
# dense vs sparse - small example
from scipy.sparse import csr_matrix

dense = np.array([[0, 0, 1, 0, 0],
                  [0, 0, 0, 0, 0],
                  [0, 2, 0, 0, 0]])
sparse = csr_matrix(dense)

print("DENSE (stores all 15 numbers):\n", dense)
print("\nSPARSE (stores only the 2 non-zero ones):\n", sparse)
print("\nBack to dense:\n", sparse.toarray())

DENSE (stores all 15 numbers):
 [[0 0 1 0 0]
 [0 0 0 0 0]
 [0 2 0 0 0]]

SPARSE (stores only the 2 non-zero ones):
 <Compressed Sparse Row sparse matrix of dtype 'int64'
	with 2 stored elements and shape (3, 5)>
  Coords	Values
  (0, 2)	1
  (2, 1)	2

Back to dense:
 [[0 0 1 0 0]
 [0 0 0 0 0]
 [0 2 0 0 0]]


In [81]:
# bag of words - by hand on the 3 small documents
toy_docs = ["great movie great acting", "boring movie", "great story boring acting"]
toy_vocab = sorted(set(" ".join(toy_docs).split()))

def bow_vector(doc, vocab):
    words = doc.split()
    return [words.count(w) for w in vocab]

toy_bow = [bow_vector(d, toy_vocab) for d in toy_docs]
print(pd.DataFrame(toy_bow, index=["D1", "D2", "D3"], columns=toy_vocab))

    acting  boring  great  movie  story
D1       1       0      2      1      0
D2       0       1      0      1      0
D3       1       1      1      0      1


In [82]:
# bag of words with sklearn countvectorizer
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer()
toy_matrix = cv.fit_transform(toy_docs)

print("Vocabulary:", cv.get_feature_names_out())
print("Word positions:", cv.vocabulary_)
print(pd.DataFrame(toy_matrix.toarray(), index=["D1", "D2", "D3"], columns=cv.get_feature_names_out()))

Vocabulary: ['acting' 'boring' 'great' 'movie' 'story']
Word positions: {'great': 2, 'movie': 3, 'acting': 0, 'boring': 1, 'story': 4}
    acting  boring  great  movie  story
D1       1       0      2      1      0
D2       0       1      0      1      0
D3       1       1      1      0      1


In [83]:
# binary bag of words - only presence (1) or absence (0)
cv_bin = CountVectorizer(binary=True)
print(pd.DataFrame(cv_bin.fit_transform(toy_docs).toarray(), index=["D1", "D2", "D3"], columns=cv_bin.get_feature_names_out()))

    acting  boring  great  movie  story
D1       1       0      1      1      0
D2       0       1      0      1      0
D3       1       1      1      0      1


In [84]:
# bag of words - unknown words don't crash
new_doc = ["amazing movie great great"]
print(pd.DataFrame(cv.transform(new_doc).toarray(), index=["new"], columns=cv.get_feature_names_out()))

     acting  boring  great  movie  story
new       0       0      2      1      0


In [85]:
# bag of words on the 500 reviews
docs = sample["lemmatized"].fillna("")[:500].tolist()

cv500 = CountVectorizer()
bow_500 = cv500.fit_transform(docs)
words = cv500.get_feature_names_out()

print("BoW shape:", bow_500.shape)
print("Our manual vocabulary size:", len(set(" ".join(docs).split())))

total_cells = bow_500.shape[0] * bow_500.shape[1]
print(f"Non-zero cells: {bow_500.nnz:,} out of {total_cells:,} ({bow_500.nnz / total_cells:.2%})")

BoW shape: (500, 11446)
Our manual vocabulary size: 11480
Non-zero cells: 51,112 out of 5,723,000 (0.89%)


In [86]:
# top words in review 0
row = bow_500[0].toarray()[0]
for i in row.argsort()[::-1][:10]:
    print(f"{words[i]:<12} {row[i]}")

really       2
even         2
rat          1
recommend    1
full         1
furry        1
rent         1
comedy       1
gang         1
go           1


In [87]:
# most common words across all 500 reviews
word_totals = np.asarray(bow_500.sum(axis=0)).ravel()
print([(words[i], int(word_totals[i])) for i in word_totals.argsort()[::-1][:20]])

[('not', 1357), ('film', 1023), ('movie', 1009), ('one', 577), ('like', 474), ('see', 473), ('make', 426), ('do', 418), ('get', 398), ('good', 372), ('well', 360), ('time', 343), ('watch', 312), ('character', 290), ('would', 272), ('story', 270), ('go', 269), ('show', 267), ('no', 265), ('really', 254)]


In [88]:
# which words appear more in positive vs negative reviews
labels = sample["sentiment"][:500].values

pos_counts = np.asarray(bow_500[labels == "positive"].sum(axis=0)).ravel()
neg_counts = np.asarray(bow_500[labels == "negative"].sum(axis=0)).ravel()
diff = pos_counts - neg_counts

print("More in POSITIVE reviews:", [words[i] for i in diff.argsort()[::-1][:15]])
print("\nMore in NEGATIVE reviews:", [words[i] for i in diff.argsort()[:15]])

More in POSITIVE reviews: ['great', 'film', 'love', 'time', 'show', 'take', 'life', 'see', 'story', 'play', 'man', 'first', 'performance', 'young', 'two']

More in NEGATIVE reviews: ['bad', 'not', 'no', 'do', 'could', 'nothing', 'waste', 'look', 'get', 'say', 'poor', 'stupid', 'can', 'try', 'even']


In [89]:
# limiting the vocabulary to the top 1000 words
cv_small = CountVectorizer(max_features=1000)
print("Limited BoW shape:", cv_small.fit_transform(docs).shape)

Limited BoW shape: (500, 1000)


In [90]:
# bigrams - fixing the "not good" problem
cv_ng = CountVectorizer(ngram_range=(1, 2))
neg_docs = ["movie not good", "movie good"]
print(pd.DataFrame(cv_ng.fit_transform(neg_docs).toarray(), index=["A", "B"], columns=cv_ng.get_feature_names_out()))

   good  movie  movie good  movie not  not  not good
A     1      1           0          1    1         1
B     1      1           1          0    0         0


In [91]:
# one-hot vs bow - same document, side by side
toy_vocab = ["acting", "boring", "great", "movie", "story"]

def one_hot(word):
    return [1 if v == word else 0 for v in toy_vocab]

for doc in ["great movie great acting", "boring movie"]:
    oh = np.array([one_hot(w) for w in doc.split()])
    bow = oh.sum(axis=0)          # add the one-hot rows together
    print(f"\n'{doc}'")
    print("one-hot shape:", oh.shape, "->", oh.size, "numbers")
    print(oh)
    print("bow:", bow, "-> always", bow.size, "numbers")


'great movie great acting'
one-hot shape: (4, 5) -> 20 numbers
[[0 0 1 0 0]
 [0 0 0 1 0]
 [0 0 1 0 0]
 [1 0 0 0 0]]
bow: [1 0 2 1 0] -> always 5 numbers

'boring movie'
one-hot shape: (2, 5) -> 10 numbers
[[0 1 0 0 0]
 [0 0 0 1 0]]
bow: [0 1 0 1 0] -> always 5 numbers
